In [ ]:
!pip install -qU  unsloth
!pip install -qU  trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 607.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 21.8 MB/s eta 0:00:00


In [ ]:
# !pip uninstall -y trl

Found existing installation: trl 0.23.0
Uninstalling trl-0.23.0:
  Successfully uninstalled trl-0.23.0


In [ ]:
# !pip cache purge

Files removed: 84


### Generating synthetic data

In [ ]:
import pandas as pd
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
from datasets import Dataset
import json
from sklearn.utils import resample

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [ ]:
DATASET_PATH = "/content/o6ai-agent-hr/data/resume_evaluation_dataset-3000.json"

print(f"Loading new dataset from {DATASET_PATH}...")
df = pd.read_json(DATASET_PATH)
print(f"✅ Loaded {len(df)} new training samples.")

# --- Verify the new dataset is balanced ---
print("\nNew dataset distribution:")
# Create the 'simple_status' column for analysis
df['simple_status'] = df['status'].apply(lambda x: 'SELECTED' if x in ['APPROVE', 'STRONGLY_CONSIDER', 'CONSIDER'] else 'REJECTED')
print(df['simple_status'].value_counts())

Loading new dataset from /content/o6ai-agent-hr/data/resume_evaluation_dataset-3000.json...
✅ Loaded 3000 new training samples.

New dataset distribution:
simple_status
REJECTED    1500
SELECTED    1500
Name: count, dtype: int64


In [ ]:
df_train = df.sample(frac=0.9, random_state=42)
df_eval = df.drop(df_train.index)

print(f"\nTraining on {len(df_train)} samples, evaluating on {len(df_eval)} samples.")


Training on 2700 samples, evaluating on 300 samples.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
)

==((====))==  Unsloth 2025.9.9: Fast Qwen2 patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.9.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [ ]:
hf_sft_dataset_train = Dataset.from_pandas(df_train)
hf_sft_dataset_eval = Dataset.from_pandas(df_eval)

def format_for_simplified_sft(example):
    """Formats an example for the simplified SFT task (score and status only)."""
    try:
        evaluation_data = json.loads(example['evaluation_json'])
        score = evaluation_data.get('overall_score', 0)
        candidate_skills = evaluation_data.get('skills_match', {}).get('present_skills', [])
        candidate_skills_str = ", ".join(candidate_skills)
    except (json.JSONDecodeError, TypeError):
        return {"text": ""}

    simple_status = "SELECTED" if example['status'] in ['APPROVE', 'STRONGLY_CONSIDER', 'CONSIDER'] else "REJECTED"

    user_prompt = (
        f"You are an HR expert. Evaluate the candidate's skills against the job requirements.\n\n"
        f"### Job Requirements:\n{example['job_description_skills']}\n\n"
        f"### Candidate Skills:\n{candidate_skills_str}\n\n"
        f"Provide your assessment in a simple JSON format containing only the status and score."
    )

    assistant_response = f'{{"score": {score}, "status": "{simple_status}"}}'

    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Apply formatting
formatted_train = hf_sft_dataset_train.map(format_for_simplified_sft)
formatted_eval = hf_sft_dataset_eval.map(format_for_simplified_sft)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
output_model_dir = "/content/o6ai-agent-hr/models/sft_v4_large_balanced_model"

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = output_model_dir,
    report_to = "wandb",
    # --- FIX: Align evaluation and save strategies ---
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps", # Changed from "epoch" to "steps"
    save_steps=20,         # Save every 20 steps to match eval
    load_best_model_at_end=True,
)

# wandb.init(project="o6ai-resume-evaluation-SFT", name="sft-v4-large-balanced-run", reinit=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_train,
    eval_dataset = formatted_eval,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = training_args,
)

print("\n--- [SFT v4] Starting training on new large, balanced dataset... ---")
trainer.train()

print(f"\n✅ [SFT v4] Training complete. Best model saved to {output_model_dir}")
# wandb.finish()



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



--- [SFT v4] Starting training on new large, balanced dataset... ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 2 | Total steps = 676
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreyanshjaino6ai (o6ailabs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
20,0.407000,0.330570
40,0.321800,0.312044
60,0.307600,0.305275
80,0.307200,0.302419
100,0.303600,0.303446
120,0.300500,0.299139
140,0.305500,0.296825
160,0.303700,0.295217
180,0.296100,0.296420
200,0.296400,0.292735


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ [SFT v4] Training complete. Best model saved to /content/o6ai-agent-hr/models/sft_v4_large_balanced_model
